# Collecting more metadata

In MGni.py there are helpers for collecting metdata from [MGnify](https://www.ebi.ac.uk/metagenomics/) or [BioSamples](https://www.ebi.ac.uk/biosamples/) for a list of MGnify accessions. 

On this page we will learn how to: 
- **Collect metadata** from MGnify using `mgnipy.collect.MGnetizer`
- **Collect metadata** from BioSamples using `mgnipy.collect.BioSampler`

This is especially useful if you already know the list of MGnify items that you would like the detailed metadata for such as a list of study accessions. Additionally, when you already have a MGnify dataset of samples and would like to get more metadata starting from the Run accessions which we will demonstrate below. 

```{margin}
After clicking the "Activate Notebook" button you can run the cells in this browser. Alternatively, you can also click on the 🚀 to launch in colab or binder.
```
<button title="Make live" style="display:inline-flex;align-items:center;gap:0.4rem;padding:0.5rem 1rem;border:0;border-radius:20px;background:linear-gradient(135deg,#0f766e,#14b8a6);color:white;cursor:pointer;font-size:1rem;" class="thebe-button" onclick="initThebeSBT()">Activate Notebook</button>

---

In [1]:
# uncomment below if colab
#!pip install mgnipy

We'll pick up from the previous page where we had downloaded the "ERP014435_GO-slim_abundances_v3.0.tsv" dataset from MGnify. 

In [2]:
import pandas as pd

# read in GO-slim abundances file
df_go = pd.read_csv("downloads/ERP014435_GO-slim_abundances_v3.0.tsv", sep="\t")

# get run accessions as a list
run_ids = df_go.columns[3:].to_list()

# sanity check
df_go.head()

,GO,description,category,ERR1299314,ERR1299315,ERR1299316,ERR1299317,ERR1299319,ERR1299320,ERR1299321,ERR1299323,ERR1299324,ERR1299325,ERR1299326,ERR1299330,ERR1299331,ERR1299333
0,GO:0000015,phosphopyruvate hydratase complex,cellular component,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,GO:0000150,recombinase activity,molecular function,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,GO:0000160,phosphorelay signal transduction system,biological process,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,GO:0000166,nucleotide binding,molecular function,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,GO:0003674,molecular function,molecular function,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## The `MGnetizer`

The run accessions/ids can be passed to a `MGnetizer` to collect their detailed metadata. MGnetizer's are a lot like MGnifiers:
- they can be accessed as attributes from MGnipy client, inheriting the configuration
- they build the set of queries lazily which you can explore via `.explain()` before executing them

In [3]:
from mgnipy import MGnipy

# init client
MG = MGnipy(cache_dir=None)

# init mgnetizer
mnet = MG.mgnetizer(resource="run", all_ids=run_ids)

# check out query set
mnet.explain()

https://www.ebi.ac.uk/metagenomics/api/v2/runs/ERR1299314
https://www.ebi.ac.uk/metagenomics/api/v2/runs/ERR1299315
https://www.ebi.ac.uk/metagenomics/api/v2/runs/ERR1299316
https://www.ebi.ac.uk/metagenomics/api/v2/runs/ERR1299317
https://www.ebi.ac.uk/metagenomics/api/v2/runs/ERR1299319
https://www.ebi.ac.uk/metagenomics/api/v2/runs/ERR1299320
https://www.ebi.ac.uk/metagenomics/api/v2/runs/ERR1299321
https://www.ebi.ac.uk/metagenomics/api/v2/runs/ERR1299323
https://www.ebi.ac.uk/metagenomics/api/v2/runs/ERR1299324
https://www.ebi.ac.uk/metagenomics/api/v2/runs/ERR1299325
https://www.ebi.ac.uk/metagenomics/api/v2/runs/ERR1299326
https://www.ebi.ac.uk/metagenomics/api/v2/runs/ERR1299330
https://www.ebi.ac.uk/metagenomics/api/v2/runs/ERR1299331
https://www.ebi.ac.uk/metagenomics/api/v2/runs/ERR1299333


now actually executing the above with `.enrich()` or `.aenrich()`

In [4]:
with mnet:
    mnet.enrich()

Enriching metadata from MGnify: 100%|██████████| 14/14 [00:01<00:00, 11.75it/s]


again we can access the metadata via `.metadata`

In [5]:
# as df
run_md = mnet.metadata.to_pandas(expand_nested_dicts=False)
# check it out
run_md.head()

,experiment_type,instrument_model,instrument_platform,sample,study,accession,sample_accession,study_accession
0,Metatranscriptomic,None,None,"{'accession': 'SAMEA3886583', 'ena_accessions'...","{'accession': 'MGYS00001374', 'ena_accessions'...",ERR1299314,SAMEA3886583,MGYS00001374
1,Metatranscriptomic,None,None,"{'accession': 'SAMEA3886584', 'ena_accessions'...","{'accession': 'MGYS00001374', 'ena_accessions'...",ERR1299315,SAMEA3886584,MGYS00001374
2,Metatranscriptomic,None,None,"{'accession': 'SAMEA3886585', 'ena_accessions'...","{'accession': 'MGYS00001374', 'ena_accessions'...",ERR1299316,SAMEA3886585,MGYS00001374
3,Metatranscriptomic,None,None,"{'accession': 'SAMEA3886586', 'ena_accessions'...","{'accession': 'MGYS00001374', 'ena_accessions'...",ERR1299317,SAMEA3886586,MGYS00001374
4,Metatranscriptomic,None,None,"{'accession': 'SAMEA3886588', 'ena_accessions'...","{'accession': 'MGYS00001374', 'ena_accessions'...",ERR1299319,SAMEA3886588,MGYS00001374


## The `BioSampler`

The above sample accessions can be passed to a `BioSampler` to collect even more metadata from the [BioSamples](https://www.ebi.ac.uk/biosamples/) database.

BioSamplers can also be accessed from the MGnipy instance, inheriting config.

In [6]:
bios = MG.biosampler(sample_ids=run_md["sample_accession"].to_list())
print(bios)

BioSampler with 14 sample_ids. 
Progress: 0 ids. 
Cache directory: None 



Note: if wanting to pass runs accessions above instead (e.g., `MG.biosampler(sample_ids=run_ids)`) then `.enrich(incl_ena=True)`

now that we have built the queries we can execute them 

In [7]:
with bios:
    bios.enrich()

Enriching biosamples: 100%|██████████| 14/14 [00:09<00:00,  1.49it/s]


In [8]:
bios.metadata.to_pandas(expand_nested_dicts=False).head()

,GivenID,SampleID,RunID,SRA accession,name,taxid,ENA-CHECKLIST,ENA-FIRST-PUBLIC,ENA-LAST-UPDATE,External Id,...,geographic location (latitude),geographic location (longitude),investigation type,organism,project name,scientific_name,sequencing method,soil environmental package,title,description
0,SAMEA3886583,SAMEA3886583,None,ERS1073717,I9,256318,ERC000022,2016-03-06T17:05:13Z,2016-10-21T09:34:22Z,SAMEA3886583,...,62.182784,50.550976,metagenome,metagenome,PRJEB12905,metagenome,Illumina 2000,soil,Birch,NaN
1,SAMEA3886584,SAMEA3886584,None,ERS1073718,I10,256318,ERC000022,2016-03-06T17:05:13Z,2016-10-21T09:34:22Z,SAMEA3886584,...,62.182784,50.550976,metagenome,metagenome,PRJEB12905,metagenome,Illumina 2000,soil,Birch,Rhizosphere sample
2,SAMEA3886585,SAMEA3886585,None,ERS1073719,I7,256318,ERC000022,2016-03-06T17:05:13Z,2016-10-21T09:34:22Z,SAMEA3886585,...,62.182785,50.550977,metagenome,metagenome,PRJEB12905,metagenome,Illumina 2000,soil,Birch,NaN
3,SAMEA3886586,SAMEA3886586,None,ERS1073720,I8,256318,ERC000022,2016-03-06T17:05:13Z,2016-10-21T09:34:22Z,SAMEA3886586,...,62.182785,50.550977,metagenome,metagenome,PRJEB12905,metagenome,Illumina 2000,soil,Birch,Rhizosphere sample
4,SAMEA3886588,SAMEA3886588,None,ERS1073722,I12,256318,ERC000022,2016-03-06T17:05:13Z,2016-10-21T09:34:22Z,SAMEA3886588,...,62.152829,50.392303,metagenome,metagenome,PRJEB12905,metagenome,Illumina 2000,soil,Birch,Rhizosphere sample


From here of course you can take over to merge the sets of MGnify metadata and Biosamples metadata. 

However mgnipy has a helper class that combines a MGnify dataset with its metadata:

---

## `MTG` MGic (the) Gatherer
The MGic gatherer (MTG) takes a dataset as pandas or polars dataframe and MGnify or BioSamples metadata and combines them into a single object. 

MTG can be used to enrich the dataset with metadata, and to convert the dataset into different formats such as pandas, polars, or anndata.


In [9]:
MTG = MG.mtg(
    dataset=df_go,
    var_cols=["description", "category"],
    var_index="GO",
    obs_index="name_of_your_chosing",
    # mgnify_runs=mnet.metadata.to_list() #can pass here or assign the sets later
)

# can assign the sets at any time after init
MTG.mgnify_runs = mnet.metadata.to_list()
MTG.biosamples_metadata = bios.metadata.to_list()

# info
print(MTG)

MTG containing:
- Dataset type: <class 'pandas.core.frame.DataFrame'>
- var_cols: ['description', 'category']
- var_index: 'GO'
- obs_index: 'name_of_your_chosing'
- Nonempty metadata sets: .mgnify_runs, .biosamples_metadata



### Example 1. to `polars`

In [10]:
# the original but as a polars df
# MTG.to_polars()

# the feature matrix
# MTG.X(df_engine="polars") # default is pandas

# the features metadata
# MTG.var_metadata(df_engine="polars")

# the obs (samples) metadata
MTG.obs_metadata(df_engine="polars")

name_of_your_chosing,experiment_type,instrument_model,instrument_platform,sample_accession,study_accession,sample__accession,sample__ena_accessions,sample__sample_title,sample__biome,sample__updated_at,study__accession,study__ena_accessions,study__title,study__updated_at,study__biome.biome_name,study__biome.lineage,GivenID,RunID,SRA accession,name,taxid,ENA-CHECKLIST,ENA-FIRST-PUBLIC,ENA-LAST-UPDATE,External Id,INSDC center name,INSDC first public,INSDC last update,INSDC status,Submitter Id,collection date,depth,environment (biome),environment (feature),environment (material),geographic location (country and/or sea),geographic location (elevation),geographic location (latitude),geographic location (longitude),investigation type,organism,project name,scientific_name,sequencing method,soil environmental package,title,description
str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""ERR1299314""","""Metatranscriptomic""",null,null,"""SAMEA3886583""","""MGYS00001374""","""SAMEA3886583""","[""ERS1073717"", ""SAMEA3886583""]","""Birch""",null,"""2026-04-28T04:56:14.026000+00:…","""MGYS00001374""","[""ERP014435"", ""PRJEB12905""]","""Ectomycorrhizal and fine root …","""2026-05-28T15:46:51.301000+00:…","""Forest soil""","""root:Host-associated:Plants:Rh…","""SAMEA3886583""",null,"""ERS1073717""","""I9""",256318,"""ERC000022""","""2016-03-06T17:05:13Z""","""2016-10-21T09:34:22Z""","""SAMEA3886583""","""UNIVERSITY OF TARTU""","""2016-03-06T17:05:13Z""","""2016-10-21T09:34:22Z""","""public""","""I9""","""2009""","""0.2""","""forest""","""birch stand""","""soil""","""Russia""","""130""","""62.182784""","""50.550976""","""metagenome""","""metagenome""","""PRJEB12905""","""metagenome""","""Illumina 2000""","""soil""","""Birch""",null
"""ERR1299315""","""Metatranscriptomic""",null,null,"""SAMEA3886584""","""MGYS00001374""","""SAMEA3886584""","[""SAMEA3886584"", ""ERS1073718""]","""Birch""",null,"""2026-04-28T04:56:11.949000+00:…","""MGYS00001374""","[""ERP014435"", ""PRJEB12905""]","""Ectomycorrhizal and fine root …","""2026-05-28T15:46:51.301000+00:…","""Forest soil""","""root:Host-associated:Plants:Rh…","""SAMEA3886584""",null,"""ERS1073718""","""I10""",256318,"""ERC000022""","""2016-03-06T17:05:13Z""","""2016-10-21T09:34:22Z""","""SAMEA3886584""","""UNIVERSITY OF TARTU""","""2016-03-06T17:05:13Z""","""2016-10-21T09:34:22Z""","""public""","""I10""","""2009""","""0.2""","""forest""","""birch stand""","""soil""","""Russia""","""130""","""62.182784""","""50.550976""","""metagenome""","""metagenome""","""PRJEB12905""","""metagenome""","""Illumina 2000""","""soil""","""Birch""","""Rhizosphere sample"""
"""ERR1299316""","""Metatranscriptomic""",null,null,"""SAMEA3886585""","""MGYS00001374""","""SAMEA3886585""","[""ERS1073719"", ""SAMEA3886585""]","""Birch""",null,"""2026-04-28T04:56:13.206000+00:…","""MGYS00001374""","[""ERP014435"", ""PRJEB12905""]","""Ectomycorrhizal and fine root …","""2026-05-28T15:46:51.301000+00:…","""Forest soil""","""root:Host-associated:Plants:Rh…","""SAMEA3886585""",null,"""ERS1073719""","""I7""",256318,"""ERC000022""","""2016-03-06T17:05:13Z""","""2016-10-21T09:34:22Z""","""SAMEA3886585""","""UNIVERSITY OF TARTU""","""2016-03-06T17:05:13Z""","""2016-10-21T09:34:22Z""","""public""","""I7""","""2009""","""0.2""","""forest""","""birch stand""","""soil""","""Russia""","""130""","""62.182785""","""50.550977""","""metagenome""","""metagenome""","""PRJEB12905""","""metagenome""","""Illumina 2000""","""soil""","""Birch""",null
"""ERR1299317""","""Metatranscriptomic""",null,null,"""SAMEA3886586""","""MGYS00001374""","""SAMEA3886586""","[""SAMEA3886586"", ""ERS1073720""]","""Birch""",null,"""2026-04-28T04:56:17.200000+00:…","""MGYS00001374""","[""ERP014435"", ""PRJEB12905""]","""Ectomycorrhizal and fine root …","""2026-05-28T15:46:51.301000+00:…","""Forest soil""","""root:Host-

### Example 2. to `anndata`

as an annotated dataframe which keeps data matrices aligned with the corresponding metadata -- even when transforming the data so that there are added matrix layers.

In [11]:
# to anndata object
an_df = MTG.to_anndata()

# the feature matrix
# an_df.to_df() # or an_df.X

# the features metadata
# an_df.var

# the obs (samples) metadata
an_df.obs

,experiment_type,instrument_model,instrument_platform,sample_accession,study_accession,sample__accession,sample__ena_accessions,sample__sample_title,sample__biome,sample__updated_at,...,geographic location (latitude),geographic location (longitude),investigation type,organism,project name,scientific_name,sequencing method,soil environmental package,title,description
name_of_your_chosing,,,,,,,,,,,,,,,,,,,,,
ERR1299314,Metatranscriptomic,None,None,SAMEA3886583,MGYS00001374,SAMEA3886583,"[ERS1073717, SAMEA3886583]",Birch,None,2026-04-28T04:56:14.026000+00:00,...,62.182784,50.550976,metagenome,metagenome,PRJEB12905,metagenome,Illumina 2000,soil,Birch,None
ERR1299315,Metatranscriptomic,None,None,SAMEA3886584,MGYS00001374,SAMEA3886584,"[SAMEA3886584, ERS1073718]",Birch,None,2026-04-28T04:56:11.949000+00:00,...,62.182784,50.550976,metagenome,metagenome,PRJEB12905,metagenome,Illumina 2000,soil,Birch,Rhizosphere sample
ERR1299316,Metatranscriptomic,None,None,SAMEA3886585,MGYS00001374,SAMEA3886585,"[ERS1073719, SAMEA3886585]",Birch,None,2026-04-28T04:56:13.206000+00:00,...,62.182785,50.550977,metagenome,metagenome,PRJEB12905,metagenome,Illumina 2000,soil,Birch,None
ERR1299317,Metatranscriptomic,None,None,SAMEA3886586,MGYS00001374,SAMEA3886586,"[SAMEA3886586, ERS1073720]",Birch,None,2026-04-28T04:56:17.200000+00:00,...,62.182785,50.550977,metagenome,metagenome,PRJEB12905,metagenome,Illumina 2000,soil,Birch,Rhizosphere sample
ERR1299319,Metatranscriptomic,None,None,SAMEA3886588,MGYS00001374,SAMEA3886588,"[ERS1073722, SAMEA3886588]",Birch,None,2026-04-28T04:56:14.818000+00:00,...,62.152829,50.392303,metagenome,metagenome,PRJEB12905,metagenome,Illumina 2000,soil,Birch,Rhizosphere sample
ERR1299320,Metatranscriptomic,None,None,SAMEA3886589,MGYS00001374,SAMEA3886589,"[ERS1073723, SAMEA3886589]",Birch,None,2026-04-28T04:56:16.012000+00:00,...,66.2,26.4,metagenome,metagenome,PRJEB12905,metagenome,Illumina 2000,soil,Birch,None
ERR1299321,Metatranscriptomic,None,None,SAMEA3886590,MGYS00001374,SAMEA3886590,"[ERS1073724, SAMEA3886590]",Birch,None,2026-04-28T04:56:09.828000+00:00,...,66.2,26.4,metagenome,metagenome,PRJEB12905,metagenome,Illumina 2000,soil,Birch,Rhizosphere sample
ERR1299323,Metatranscriptomic,None,None,SAMEA3886592,MGYS00001374,SAMEA3886592,"[SAMEA3886592, ERS1073726]",Birch,None,2026-04-28T04:56:14.425000+00:00,...,61.49,29.19,metagenome,metagenome,PRJEB12905,metagenome,Illumina 2000,soil,Birch,Rhizosphere sample
ERR1299324,Metatranscriptomic,None,None,SAMEA3886593,MGYS00001374,SAMEA3886593,"[ERS1073727, SAMEA3886593]",Birch,None,2026-04-28T04:56:15.602000+00:00,...,61.1359,21.283455,metagenome,metagenome,PRJEB12905,metagenome,Illumina 2000,soil,Birch,None


In [12]:
# exporting to h5ad file
fname = "example_collectors.h5ad"
an_df.obs = an_df.obs.astype(
    str
)  # workaround for h5ad export issue with mixed types in obs
an_df.write_h5ad(fname)

/Users/anglup/GitHub/mgnipy/.venv/lib/python3.11/site-packages/anndata/_io/utils.py:272: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)


In [13]:
import anndata as ad

# read in data
back = ad.read_h5ad(fname)
# check it out
back

AnnData object with n_obs × n_vars = 14 × 116
    obs: 'experiment_type', 'instrument_model', 'instrument_platform', 'sample_accession', 'study_accession', 'sample__accession', 'sample__ena_accessions', 'sample__sample_title', 'sample__biome', 'sample__updated_at', 'study__accession', 'study__ena_accessions', 'study__title', 'study__updated_at', 'study__biome.biome_name', 'study__biome.lineage', 'GivenID', 'RunID', 'SRA accession', 'name', 'taxid', 'ENA-CHECKLIST', 'ENA-FIRST-PUBLIC', 'ENA-LAST-UPDATE', 'External Id', 'INSDC center name', 'INSDC first public', 'INSDC last update', 'INSDC status', 'Submitter Id', 'collection date', 'depth', 'environment (biome)', 'environment (feature)', 'environment (material)', 'geographic location (country and/or sea)', 'geographic location (elevation)', 'geographic location (latitude)', 'geographic location (longitude)', 'investigation type', 'organism', 'project name', 'scientific_name', 'sequencing method', 'soil environmental package', 'title', '

---

## Wrap Up:

We started with only a MGnify dataset that included a list of run accessions. 

This page was a quick start demonstration of:

1. ✅ Using `MGnetizer`s to collect metadata from MGnify

2. ✅ Collecting even more metadata from BioSamples with `BioSampler`

3. ✅ Merging the dataset with the rich metadata using `MTG`